# 🛡️ PharmaGuard — AI Pharmacy Security Monitor
### Cloud-Assisted Real-Time Suspicious-Interaction Monitoring System

---

## 1. Introduction

**PharmaGuard** is an AI-assisted real-time surveillance system designed for pharmacy operations. It monitors designated pharmacy shelf areas (e.g. prescription medicines or controlled substances), tracks individuals across frames, detects suspicious behavioral interaction sequences (such as prolonged dwell/loitering or rapid grab-and-retreat movement), and alerts pharmacy staff for human review.

### Cloud-Assisted Architecture
The system employs a browser-to-cloud computer vision pipeline:
```
[ Laptop Webcam ]
        │  (browser captures 640x480 JPEG frames, one in flight at a time)
        ▼
[ Secure HTTPS Live Tunnel ] (Gradio .live tunnel)
        │
        ▼
[ Google Colab GPU Cloud Runtime ]
        ├─► [ Person Detection ]  (Pretrained YOLOv8 / RT-DETR)
        ├─► [ Multi-Object MOT ]  (IoU tracker, ByteTrack-style)
        ├─► [ Behavior Analysis ] (Shelf ROI Dwell Time & Loitering)
        ├─► [ Alert Management ]  (Severity Grading & Cooldown Throttling)
        └─► [ Frame Annotation ]  (Bounding Boxes, IDs, Trails & HUD)
        │
        ▼
[ Browser Monitoring Dashboard ] (Real-Time Video + Telemetry + Alerts)
```


---

## 2. Runtime Configuration

Inspect the active Colab environment, Python runtime version, CUDA drivers, and GPU hardware allocation. A GPU runtime (NVIDIA T4, L4, V100, or A100) is strongly recommended for real-time inference.


In [ ]:
import sys
import platform
import torch

print("=" * 65)
print("🔍 PHARMAGUARD RUNTIME ENVIRONMENT CHECK")
print("=" * 65)

# 1. Python Environment
print(f"Python Version:   {platform.python_version()} ({sys.executable})")

# 2. PyTorch & CUDA Availability
print(f"PyTorch Version:  {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available:   {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cuda_version = torch.version.cuda
    print(f"Active GPU Model: {gpu_name} ({total_mem_gb:.2f} GB VRAM)")
    print(f"Allocated GPUs:   {gpu_count}")
    print(f"CUDA Toolkit:     {cuda_version}")
    print("\n✅ GPU hardware detected. Real-time inference acceleration enabled.")
else:
    print("\n⚠️  WARNING: NO GPU DETECTED!")
    print("   Running on CPU will limit processing throughput to ~8-10 FPS.")
    print("   To switch to GPU: Click Runtime > Change runtime type > Select T4 GPU > Save.")
print("=" * 65)


---

## 3. Repository Setup

Clone the official PharmaGuard GitHub repository into `/content/pharmguard` and verify that all core modules exist. If the directory already exists, it pulls the latest updates from remote.


In [ ]:
import base64
import os
import subprocess

# Option 1 runs the feat/phone-cameras branch (rear-camera default, access key, Option 2 code).
# Private repo: add a GitHub token as Colab Secret GITHUB_TOKEN (read-only, Contents: Read). It is sent as a
# request header only, never written into .git/config or printed (same method as O2-3).
REPO_URL = "https://github.com/Franklinnn099/Pharmguard.git"
BRANCH = "feat/phone-cameras"
TARGET_DIR = "/content/pharmguard"

print("=" * 65)
print("📁 PHARMAGUARD REPOSITORY INITIALIZATION")
print("=" * 65)

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass   # no secret / no notebook access: public clone
auth = ["-c", "http.extraHeader=Authorization: Basic "
        + base64.b64encode(f"x-access-token:{token}".encode()).decode()] if token else []

def git(*args):
    res = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    if res.returncode != 0:
        step = next(a for a in args if not a.startswith("-") and a != TARGET_DIR)
        raise SystemExit(f"❌ git {step} failed (private repo? add the GITHUB_TOKEN secret):\n{res.stderr.strip()}")
    return res.stdout.strip()

if os.path.isdir(os.path.join(TARGET_DIR, ".git")):
    print(f"Repository already at {TARGET_DIR}; updating to {BRANCH}...")
    git("-C", TARGET_DIR, "fetch", "origin", BRANCH)
    git("-C", TARGET_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD")
elif os.path.exists("pharmaguard") and os.path.exists("main.py"):
    TARGET_DIR = os.getcwd()
    print(f"Already running from project directory: {TARGET_DIR}")
else:
    print(f"Cloning {BRANCH} from {REPO_URL} into {TARGET_DIR}...")
    git("clone", "--branch", BRANCH, REPO_URL, TARGET_DIR)
if not os.path.isdir(TARGET_DIR):
    raise SystemExit(f"❌ {TARGET_DIR} does not exist after the clone; see the error above.")
os.chdir(TARGET_DIR)
head = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"Code: {TARGET_DIR} @ {branch} -> {head} | token used: {bool(token)}")

# Verify presence of all required project files
required_files = [
    "main.py",
    "benchmark.py",
    "requirements.txt",
    "pharmaguard/__init__.py",
    "pharmaguard/config.py",
    "pharmaguard/detector.py",
    "pharmaguard/tracker.py",
    "pharmaguard/behavior.py",
    "pharmaguard/alerts.py",
    "pharmaguard/events.py",
    "pharmaguard/annotator.py",
    "pharmaguard/pipeline.py",
    "pharmaguard/ui.py",
    "pharmaguard/app.py",
]

missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    raise SystemExit(f"❌ Missing required project files: {missing_files}")
print(f"✅ All {len(required_files)} core modules and runners verified in {os.getcwd()}.")
print("=" * 65)

---

## 4. Dependency Installation

Install the pinned dependencies from `requirements.txt`.
- Handles package installations cleanly without arbitrary version changes.
- Checks compatibility of Gradio and Pydantic schemas to ensure smooth WebSocket streaming.
- Verifies critical vision and machine learning imports.


In [ ]:
import sys
import subprocess

print("=" * 65)
print("📦 INSTALLING PHARMAGUARD DEPENDENCIES")
print("=" * 65)

# Install pinned dependencies, constrained to Colab's own CUDA build of torch/torchvision (same file as O2-4)
# so pip can never replace it with a CPU or mismatched wheel.
from importlib.metadata import version as _v   # what pip compares (torch.__version__ can differ)
KEEP_TORCH = "/content/keep_torch.txt"
with open(KEEP_TORCH, "w") as f:
    f.write(f"torch=={_v('torch')}\ntorchvision=={_v('torchvision')}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-c", KEEP_TORCH], check=True)
# Check in a fresh interpreter: this one already imported torch before the install.
cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
                      capture_output=True, text=True).stdout.split()
assert cuda and cuda[-1] == "True", (
    f"torch.cuda.is_available() is False after install (torch {cuda[:1]}). Use a GPU runtime "
    "(Runtime > Change runtime type > T4 GPU) and re-run from Section 2.")
print(f"✅ CUDA torch kept: {cuda[0]}")

# Verify critical package imports and versions
import torch
import cv2
import ultralytics
import gradio as gr
import pydantic
import numpy as np

print("\nPackage Verification:")
print(f"  - PyTorch:      {torch.__version__}")
print(f"  - OpenCV:       {cv2.__version__}")
print(f"  - Ultralytics:  {ultralytics.__version__}")
print(f"  - Gradio:       {gr.__version__}")
print(f"  - Pydantic:     {pydantic.__version__}")
print(f"  - NumPy:        {np.__version__}")

# Check Gradio / Pydantic JSON-schema compatibility
try:
    import jsonschema
    print(f"  - JSONSchema:   {jsonschema.__version__}")
    print("✅ Schema serialization verified clean.")
except ImportError:
    print("ℹ️ jsonschema module optional; default validation active.")

print("=" * 65)

---

## 5. Authentication Configuration

PharmaGuard requires authentication on public tunnels to prevent unauthorized viewing of the pharmacy surveillance feed.

### How Authentication Works:
1. **Google Colab Secrets (Recommended)**: Set `PHARMAGUARD_AUTH_USER` and `PHARMAGUARD_AUTH_PASS` in Colab's left sidebar (🔑 Secrets icon) and toggle "Notebook access".
2. **Environment Variables**: Configure your desired username and password in the cell below.
3. **Automatic Ephemeral Token**: If no credentials are set, `pharmaguard/app.py` will generate a secure, random one-time password and display it upon launch.


In [ ]:
import os
import getpass

print("=" * 65)
print("🔐 CONFIGURE DASHBOARD AUTHENTICATION")
print("=" * 65)

# 1. Attempt retrieval from Google Colab Secrets
colab_user, colab_pass = None, None
try:
    from google.colab import userdata
    colab_user = userdata.get("PHARMAGUARD_AUTH_USER")
    colab_pass = userdata.get("PHARMAGUARD_AUTH_PASS")
except Exception:
    pass

if colab_user and colab_pass:
    os.environ["PHARMAGUARD_AUTH_USER"] = colab_user
    os.environ["PHARMAGUARD_AUTH_PASS"] = colab_pass
    print(f"✅ Loaded credentials securely from Colab Secrets (Operator: {colab_user})")
elif os.environ.get("PHARMAGUARD_AUTH_USER") and os.environ.get("PHARMAGUARD_AUTH_PASS"):
    print(f"✅ Loaded credentials from environment (Operator: {os.environ.get('PHARMAGUARD_AUTH_USER')})")
else:
    print("ℹ️  No static credentials configured.")
    print("   PharmaGuard will generate a one-time ephemeral session password on startup.")
    print("   (Optional: to configure your own credentials, set PHARMAGUARD_AUTH_USER and PHARMAGUARD_AUTH_PASS)")
print("=" * 65)


---

## 6. Model Initialization & Verification

Preload the pretrained model weights (`yolov8n-pose.pt` for people + wrist keypoints; `yolov8n.pt` is also used for bag detection), verify that the model is allocated to the GPU, and run an initial warmup inference on an empty test frame.


In [ ]:
import torch
import numpy as np
from pharmaguard.config import DetectorConfig
from pharmaguard.detector import YOLODetector

print("=" * 65)
print("🤖 INITIALIZING VISION DETECTOR")
print("=" * 65)

target_device = "cuda" if torch.cuda.is_available() else "cpu"
detector_cfg = DetectorConfig(model_name="yolov8n-pose.pt", device=target_device)  # pose: person boxes + wrist/hip keypoints

print(f"Loading {detector_cfg.model_name} onto compute device: {target_device.upper()}...")
detector = YOLODetector(detector_cfg)

model_info = detector.get_model_info()
print("Model Metadata:")
print(f"  - Weights File:        {model_info.get('model_name')}")
print(f"  - Active Compute Unit: {model_info.get('device').upper()}")
print(f"  - Confidence Filter:   {model_info.get('confidence_threshold')}")
print(f"  - Target Classes:      {model_info.get('target_classes')} (Person)")
print(f"  - Pose keypoints:      {model_info.get('has_keypoints')}")

# Warmup inference
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)
test_output = detector.detect(dummy_frame)
print(f"Warmup inference completed successfully (Detections: {test_output.count}).")
print("✅ Model loaded and ready for real-time tracking.")
print("=" * 65)


---

## 7. GPU Benchmark (server compute only)

Runs `benchmark.py` on a real photo containing people (or a video via `--video`), measuring every server-side stage
**including the JPEG return encoding** Gradio performs after the pipeline:

`downscale -> detect -> track -> behaviour -> annotate -> encode`

> ⚠️ This is compute only. It does **not** include the browser <-> Colab network path, which is usually the larger cost
> over a `gradio.live` tunnel. Measure that from your laptop with `tools/e2e_benchmark.py` (see Section 8b),
> and read the live **Delivered FPS** / **Latency breakdown** panels on the dashboard.
>
> Record the printed table in `PROJECT_STATUS.md` - these are the first real GPU numbers for the project.

In [ ]:
# Shell (!) form streams output live into this cell.
!python benchmark.py --frames 100
# Faster variant if FPS is too low:  !python benchmark.py --frames 100 --imgsz 480 --max-width 480

---

## 7b. Persistent Event Storage on Google Drive (recommended)

Mount Drive so recorded events (SQLite) and evidence clips survive the Colab session.
PharmaGuard never opens the database directly on Drive (SQLite + Drive sync = locking/corruption risk):
it writes locally and copies to `MyDrive/PharmaGuard/` with SQLite's backup API.

> 🔒 Evidence clips show people's faces. Use staged footage of consenting participants only.
> Clips and events older than `recording.retention_days` (default 7) are deleted at startup.

In [ ]:
import os
MIRROR_ARG = ""
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/PharmaGuard", exist_ok=True)
    MIRROR_ARG = "--mirror-dir /content/drive/MyDrive/PharmaGuard"
    print("✅ Drive mounted: events will be mirrored to MyDrive/PharmaGuard")
except Exception as exc:
    print(f"ℹ️  Drive not mounted ({exc}); events stay in /content/pg_data for this session only.")

---

## 8. Launch PharmaGuard Security Console

Start the live PharmaGuard surveillance server with public HTTPS tunneling enabled (`--colab`).

### Instructions to Connect Your Laptop Webcam:
1. Run the cell below.
2. Look for the public tunnel link printed in the output: **`Running on public URL: https://xxxx.gradio.live`**.
3. Click the link to open the dashboard in your laptop browser.
4. **Log in**:
   - If you configured credentials in Section 5, enter them.
   - Otherwise, copy the **`[SECURITY] Generated one-time ephemeral session credentials`** printed in the output below.
5. In the **Live monitor** tab, click **Start camera** (fast stream: several frames in flight, much smoother over the tunnel).
6. When prompted by your browser (**Allow access to your camera?**), click **Allow**.
   - **Max delay** (default 300 ms): how far the video may run behind real time. Lower = closer to live; boxes are predicted for the rest and can briefly overshoot when someone stops. **Auto** = boxes exact. **Upload** 640 px (default); try 480 or 320 px on a slow connection.
   - Detections slow? Try **Frames in flight** 4.
   - The old Gradio webcam is under *Fallback* (collapsed). Never run both at once.
7. Annotated frames return to the right-hand panel. Read the **measured** figures on the dashboard:
   - **Delivered FPS** (end-to-end, measured on the server from frame arrival spacing)
   - **Displayed FPS** (counted in your browser)
   - **Latency breakdown**: full cycle p50/p95, server compute, transport+encode

### Using the iPhone as the camera (Option 1)
1. Run the cell below and wait for **`Running on public URL: https://xxxx.gradio.live`**.
2. Open that **https** link on the iPhone in **Safari** (send it to yourself; don't use an in-app browser).
3. **Log in** with the same credentials as above (Section 5 secrets, or the generated ones printed below).
4. In the **Live monitor** tab tap **Start camera**, then tap **Allow** when Safari asks for the camera.
5. Check it is the **REAR** camera and **not mirrored**: hold up printed text - it must read normally, left to right.
   If you see your own face, the browser ignored the rear-camera request: reload the page and tap Start camera again.
6. iPhone **Settings → Display & Brightness → Auto-Lock → Never** (set it back after the demo).
7. Keep **Safari in the foreground** with the screen on: iOS pauses the camera when Safari is in the background,
   the screen locks, or another app opens. Plug the phone in.

> 🔒 **Privacy**: frames travel through Gradio's public share relay (`*.gradio.live`). Use only staged footage of
> people who have agreed to be filmed - never real pharmacy customers.

### 8b. Measure end-to-end performance from your laptop (optional, ~1 min)
Stop the browser webcam first (one stream at a time), then on your **laptop** in the repo folder run:
```
python tools/e2e_benchmark.py --url https://xxxx.gradio.live --user <user> --password <password>
```
It prints round-trip p50/p95, sustained FPS, and a per-stage split (upload / wait / download).


In [ ]:
MIRROR_ARG = globals().get("MIRROR_ARG", "")   # "" when Section 7b (Drive) was not run
# Shell (!) form: the public URL and any generated password stream into this cell's output.
# Edit configs/demo_shelf.yaml to outline your mock shelf (see docs/DEMO_STAGING.md).
# Events + evidence clips go to fast local disk (/content/pg_data); if Drive is mounted (Section 7b)
# the database and clips are also copied to Drive after every event.
!python main.py --colab --config configs/demo_shelf.yaml --data-dir /content/pg_data {MIRROR_ARG}
# Lower-latency variant: add  --imgsz 480 --max-width 480

---

## 9. Live System Verification Guide

Full scenario list, camera placement and expected results: **`docs/DEMO_STAGING.md`**.

| Test | Action | Expected result |
| :--- | :--- | :--- |
| **1. Stream up** | Start the webcam on the left. | Annotated frames appear; status `LIVE | CUDA`. |
| **2. People + IDs** | One, then two people enter. | Boxes with distinct `ID #n`; IDs survive crossing paths. |
| **3. Wrists** | Raise a hand. | Green dot on each visible wrist. |
| **3b. Zone** | **Zones** tab: Capture snapshot, click the shelf corners, Add zone, Apply, Save. | Zone appears on the live video. |
| **4. Shelf contact** | Reach an arm into the outlined shelf. | Wrist dot turns orange; zone badge shows `HAND CONTACT`. |
| **5. Browsing** | Reach in several times, look at items at chest height, put them back. | **No** review/alert. |
| **6. Staged concealment** | Reach in, then within 1-2 s put the hand into a handbag and walk away. | Amber REVIEW, then red ALARM + siren with reasons; Acknowledge silences, Reset clears. |
| **7. Alarm** | Click the page once, then **Test alarm**. | `Alarm sound: ready`; 5 s siren, purple banner/border. |
| **7b. Review** | **Event review** tab: open the alert, play the clip, mark Confirmed / False alarm. | Clip plays; false-alarm rate updates. |
| **8. Measured performance** | Stream 30 s. | Record Delivered FPS, Displayed FPS, cycle p50/p95, server ms. |
| **9. Disconnect** | Stop the webcam for 5 s. | `NO SIGNAL` within ~2 s, then recovers with `1 interruption(s)`. |

> Camera must view the actor from ~45° to the side so the reaching arm is visible (a camera directly behind the actor cannot see hands).

In [ ]:
# Record the environment this run used (paste into PROJECT_STATUS.md alongside your measurements).
import torch, gradio, ultralytics, platform
print("Python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("gradio", gradio.__version__, "| ultralytics", ultralytics.__version__)

<!-- pg-option2 -->
---

## Option 2: iPhone as the primary camera (camera server on the Colab GPU)

Sections 2-9 are **Option 1**: the dashboard's `gradio.live` link, opened on the laptop or the iPhone (rear camera by
default). This section is **Option 2**: the phone-camera server (`pharmguard-camera/`) runs here on the GPU, and a
Cloudflare quick tunnel gives it a real HTTPS address, so the iPhone just scans a QR code: no certificates, no Wi-Fi
pairing.

```
iPhone (PWA, rear camera) --wss--> https://xxxx.trycloudflare.com --> Colab: camera server + PharmaGuard pipeline (CUDA)
```

Run the cells below **in order**. They do not need Sections 3-9 (they clone their own copy into `/content/pg_cam`).

> 🔒 **Privacy**: frames pass through Cloudflare's tunnel. Use only staged footage of people who agreed to be filmed.
> The link and QR contain this session's **access key**: anyone holding them can watch the cameras. Don't share them.
>
> ⚠️ Free Colab may disconnect idle sessions or stop web UIs it detects; keep this tab open during the demo.
> Latency over the internet is higher than on the local Wi-Fi (measure it with the cells in Option 2b).

In [ ]:
# O2-1. GPU check: the camera server must run on CUDA here.
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU runtime. Runtime > Change runtime type > T4 GPU > Save, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
# O2-2. Mount Google Drive: weights, zones, alerts and evidence clips survive Colab restarts.
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/PharmaGuard"
for sub in ("weights", "zones", "camera", "logs"):
    os.makedirs(f"{DRIVE}/{sub}", exist_ok=True)
print("Drive folder:", DRIVE)

In [ ]:
# O2-3. Clone (or update) the feat/phone-cameras branch. Private repo: add a GitHub token as Colab Secret
# GITHUB_TOKEN (read-only, Contents: Read). It is sent as a header only, never written into .git/config.
import base64, os, subprocess
REPO_URL, BRANCH, CAM_DIR = "https://github.com/Franklinnn099/Pharmguard.git", "feat/phone-cameras", "/content/pg_cam"
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass   # no secret / no notebook access: public clone
auth = ["-c", "http.extraHeader=Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()] if token else []
if os.path.isdir(f"{CAM_DIR}/.git"):
    cmd = ["git", *auth, "-C", CAM_DIR, "pull", "--ff-only", "origin", BRANCH]
else:
    cmd = ["git", *auth, "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CAM_DIR]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    raise SystemExit("git failed (private repo? add the GITHUB_TOKEN secret):\n" + res.stderr)
head = subprocess.run(["git", "-C", CAM_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
SERVER_DIR = f"{CAM_DIR}/pharmguard-camera/server"
print("Code:", CAM_DIR, "@", BRANCH, "->", head, "| token used:", bool(token))

In [ ]:
# O2-4. Install the camera server's requirements WITHOUT touching Colab's CUDA build of torch/torchvision.
import subprocess, sys
from importlib.metadata import version as _v   # what pip compares (torch.__version__ can differ)
KEEP_TORCH = "/content/keep_torch.txt"
with open(KEEP_TORCH, "w") as f:
    f.write(f"torch=={_v('torch')}\ntorchvision=={_v('torchvision')}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{SERVER_DIR}/requirements.txt", "-c", KEEP_TORCH],
               check=True)
out = subprocess.run([sys.executable, "-c", "import torch, ultralytics, fastapi, qrcode; "
                      "print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)

In [ ]:
# O2-5. Storage on Drive. Weights: copied from Drive into the repo (the server prefers repo-root weights), and new
# downloads are copied back by the status/stop cells. Zones: kept on Drive (PG_ZONES_DIR). Alerts: JSONL on Drive.
# Events: SQLite on fast local disk and copied to Drive after each event with SQLite's backup API (never opened on
# Drive), evidence clips copied to Drive too.
import glob, os, shutil, yaml
for w in glob.glob(f"{DRIVE}/weights/*.pt"):
    if not os.path.exists(f"{CAM_DIR}/{os.path.basename(w)}"):
        shutil.copy2(w, CAM_DIR)

def save_weights_to_drive():
    for w in glob.glob(f"{CAM_DIR}/*.pt") + glob.glob(f"{SERVER_DIR}/*.pt"):
        if not os.path.exists(f"{DRIVE}/weights/{os.path.basename(w)}"):
            shutil.copy2(w, f"{DRIVE}/weights/")
            print("cached on Drive:", os.path.basename(w))

cfg = yaml.safe_load(open(f"{CAM_DIR}/configs/demo_shelf.yaml")) or {}
cfg.setdefault("detector", {})["device"] = "cuda"
cfg.setdefault("recording", {})["mirror_dir"] = f"{DRIVE}/camera"
PG_CONFIG = "/content/pg_cam_colab.yaml"
with open(PG_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f)
os.environ.update(PG_CONFIG=PG_CONFIG, PG_ZONES_DIR=f"{DRIVE}/zones", PHARMAGUARD_DATA_DIR="/content/pg_cam_data",
                  PHARMAGUARD_ALERT_LOG=f"{DRIVE}/logs/pharmaguard_alerts.jsonl")
print("Weights on Drive:", [os.path.basename(w) for w in glob.glob(f"{DRIVE}/weights/*.pt")] or "none yet (downloaded on first start)")
print("Zones on Drive:", os.listdir(f"{DRIVE}/zones") or "none yet (draw them at the Zones link; demo_shelf.yaml zones apply until then)")

In [ ]:
# O2-6. Access key: Colab Secret PG_ACCESS_KEY if set, otherwise a new random key for this session
# (old links and QR codes stop working when the key changes).
import os, secrets
key = None
try:
    from google.colab import userdata
    key = userdata.get("PG_ACCESS_KEY")
except Exception:
    pass
PG_ACCESS_KEY = (key or secrets.token_urlsafe(18)).strip()
os.environ["PG_ACCESS_KEY"] = PG_ACCESS_KEY
print("Access key:", "from Colab Secrets" if key else "new random key for this session")

In [ ]:
# O2-7. Start the camera server (CUDA, port 8000, localhost only: the tunnel is the only way in).
import json, os, signal, subprocess, sys, time, urllib.request
SERVER_LOG = "/content/pg_cam_server.log"
DASHBOARD_URL = globals().get("DASHBOARD_URL", "")   # set by the optional dashboard cell O2-9
server_proc = globals().get("server_proc")

def api(path, base="http://127.0.0.1:8000"):
    req = urllib.request.Request(base + path, headers={"X-PG-Key": PG_ACCESS_KEY})
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.loads(r.read())

def stop_camera_server():
    global server_proc
    if server_proc is not None and server_proc.poll() is None:
        server_proc.send_signal(signal.SIGINT)          # graceful: finishes clips, copies the event DB to Drive
        try:
            server_proc.wait(30)
        except subprocess.TimeoutExpired:
            server_proc.kill()
    server_proc = None

def start_camera_server():
    global server_proc
    stop_camera_server()
    origins = ["http://127.0.0.1:7860", "http://localhost:7860"] + ([DASHBOARD_URL.rstrip("/")] if DASHBOARD_URL else [])
    env = dict(os.environ, PG_PORT="8000", PG_HOST="127.0.0.1", PG_CORS_ORIGINS=",".join(origins), PYTHONUNBUFFERED="1")
    server_proc = subprocess.Popen([sys.executable, "-u", "main.py"], cwd=SERVER_DIR, env=env,
                                   stdout=open(SERVER_LOG, "w"), stderr=subprocess.STDOUT)
    for _ in range(300):                                  # first start downloads + loads the models
        if server_proc.poll() is not None:
            raise SystemExit("Camera server exited:\n" + open(SERVER_LOG).read()[-3000:])
        try:
            info = api("/api/cameras")
            break
        except Exception:
            time.sleep(1)
    else:
        raise SystemExit("Camera server did not answer within 300 s:\n" + open(SERVER_LOG).read()[-3000:])
    print("Camera server up | detector:", info["detector"], "|", info["model"], "| CORS:", origins)
    if "cuda" not in str(info["model"]):
        raise SystemExit("The model is NOT on the GPU: " + str(info["model"]))

start_camera_server()
save_weights_to_drive()

In [ ]:
# O2-8. Cloudflare quick tunnel (free, no account; supports WebSockets) -> public https URL for the iPhone.
import os, re, subprocess, time, urllib.request
CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
tunnel_proc = globals().get("tunnel_proc")
if tunnel_proc is not None and tunnel_proc.poll() is None:
    tunnel_proc.terminate(); tunnel_proc.wait(10)
TUNNEL_LOG = "/content/cloudflared.log"
tunnel_proc = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                               stdout=open(TUNNEL_LOG, "w"), stderr=subprocess.STDOUT)
TUNNEL_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
    if m:
        TUNNEL_URL = m.group(0); break
    time.sleep(1)
if not TUNNEL_URL:
    raise SystemExit("No tunnel URL:\n" + open(TUNNEL_LOG).read()[-2000:])
for _ in range(45):                                       # the new hostname takes a few seconds to resolve
    try:
        api("/api/cameras", base=TUNNEL_URL); break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("Tunnel URL not reachable yet: " + TUNNEL_URL + " (run this cell again)")
print("Tunnel up:", TUNNEL_URL)

In [ ]:
# O2-9. Links for the iPhone. Scan the QR with the iPhone camera, open in Safari, allow camera access.
from urllib.parse import quote
from IPython.display import HTML, display
import qrcode
CAM = "phone1"                                            # one name per phone; change it for a second phone
K = quote(PG_ACCESS_KEY)
PHONE_URL = f"{TUNNEL_URL}/?cam={CAM}&start=1&key={K}"
MONITOR_URL = f"{TUNNEL_URL}/monitor?key={K}"
ZONES_URL = f"{TUNNEL_URL}/zones/{CAM}?key={K}"
display(qrcode.make(PHONE_URL).get_image().resize((300, 300)))
display(HTML(f'<p><b>iPhone link</b> (contains the access key - do not share): <a href="{PHONE_URL}" target="_blank">{PHONE_URL}</a></p>'
             f'<p><b>Monitor</b> (laptop): <a href="{MONITOR_URL}" target="_blank">{MONITOR_URL}</a></p>'
             f'<p><b>Zone editor</b> for {CAM}: <a href="{ZONES_URL}" target="_blank">{ZONES_URL}</a></p>'))

In [ ]:
# O2-10 (optional). Also run the Gradio dashboard, with a Phone cameras tab showing these feeds.
# Starts it in the background, then restarts the camera server with the dashboard's gradio.live URL in
# PG_CORS_ORIGINS (the phone reconnects by itself). Skip this cell if /monitor is enough.
import re, subprocess, sys, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CAM_DIR}/requirements.txt", "-c", KEEP_TORCH], check=True)
DASH_LOG = "/content/pg_dashboard.log"
dash_proc = globals().get("dash_proc")
if dash_proc is not None and dash_proc.poll() is None:
    dash_proc.terminate(); dash_proc.wait(15)
dash_proc = subprocess.Popen([sys.executable, "-u", "main.py", "--colab", "--config", f"{CAM_DIR}/configs/demo_shelf.yaml",
                              "--data-dir", "/content/pg_data", "--mirror-dir", DRIVE, "--phone-cameras", TUNNEL_URL],
                             cwd=CAM_DIR, env=dict(os.environ, PYTHONUNBUFFERED="1"),
                             stdout=open(DASH_LOG, "w"), stderr=subprocess.STDOUT)
DASHBOARD_URL = None
for _ in range(240):
    log = open(DASH_LOG).read()
    m = re.search(r"https://[a-z0-9]+\.gradio\.live", log)
    if m:
        DASHBOARD_URL = m.group(0); break
    if dash_proc.poll() is not None:
        raise SystemExit("Dashboard exited:\n" + log[-3000:])
    time.sleep(1)
if not DASHBOARD_URL:
    raise SystemExit("No gradio.live URL yet:\n" + open(DASH_LOG).read()[-3000:])
print("\n".join(l for l in open(DASH_LOG).read().splitlines() if "Username" in l or "Password" in l or "SECURITY" in l))
start_camera_server()                                     # now with the dashboard origin allowed
print("Dashboard:", DASHBOARD_URL, "(Phone cameras tab)")

In [ ]:
# O2-11. Status: connected cameras, measured fps, server compute per frame, last alerts. Re-run any time.
import json, os
info = api("/api/cameras")
print("Detector:", info["detector"], "|", info["model"])
for c in info["cameras"] or []:
    a = c["alarm"] if isinstance(c["alarm"], dict) else {"state": c["alarm"]}
    print(f'  {c["camera_id"]:<10} {"LIVE" if c["connected"] else "offline":<8} {c["fps"]:>5} fps  '
          f'{c["processing_ms"]} ms/frame  {c["resolution"]:>9}  frames {c["frames"]} (skipped {c["skipped"]})  '
          f'persons {c["persons"]} flagged {c["flagged"]}  alarm {a.get("state")}')
if not info["cameras"]:
    print("  no camera has connected yet - scan the QR from O2-9")
log = os.environ["PHARMAGUARD_ALERT_LOG"]
lines = open(log).read().splitlines()[-5:] if os.path.exists(log) else []
print("Last alerts (for staff review):" if lines else "No alerts yet.")
for l in lines:
    r = json.loads(l)
    print(f'  {r.get("timestamp")}  {r.get("severity"):<8} {r.get("zone")}  {r.get("title")}')
save_weights_to_drive()

In [ ]:
# O2-12. Stop: tunnel first (no more outside access), then the dashboard and the camera server (graceful: clips
# finish and the event DB is copied to Drive).
for name in ("tunnel_proc", "dash_proc"):
    p = globals().get(name)
    if p is not None and p.poll() is None:
        p.terminate(); p.wait(15)
stop_camera_server()
save_weights_to_drive()
print("Stopped. Drive keeps weights, zones, alerts, the event DB copy and clips:", DRIVE)

<!-- pg-option2b -->
### Option 2b: measure (run after O2-8, before the demo; ~3 min)

Uses `test_client.py` (a fake phone sending a real photo of people) through the **Cloudflare tunnel**, with the
object scan on (`PG_ITEMS=1`, demo default) and off. Two runs each: 1 frame in flight (= latency) and 2 in flight
(= throughput, what the PWA does). The server restarts between settings; a connected iPhone reconnects by itself.

Send back the printed block **and** the block from M-2 run on the laptop (the real internet path to Cloudflare).

In [ ]:
# M-1. Through the tunnel, from Colab. Restarts the camera server (PG_ITEMS 1 -> 0 -> back to 1).
import os, subprocess, sys, torch
WSS = TUNNEL_URL.replace("https://", "wss://")
def bench(tag, inflight):
    tc = [sys.executable, "test_client.py", "--url", WSS, "--key", PG_ACCESS_KEY, "--quiet"]
    subprocess.run(tc + ["--frames", "10", "--cam", "warmup"], cwd=SERVER_DIR, capture_output=True)
    r = subprocess.run(tc + ["--frames", "100", "--in-flight", inflight, "--cam", f"bench{tag}{inflight}"],
                       cwd=SERVER_DIR, capture_output=True, text=True)
    keep = [l for l in r.stdout.splitlines() if "sent," in l or "round trip" in l or "FAIL" in l]
    print(f"PG_ITEMS={tag} in-flight={inflight}:\n  " + "\n  ".join(keep or [r.stderr[-500:]]))
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
for items in ("1", "0"):
    os.environ["PG_ITEMS"] = items
    start_camera_server()
    for inflight in ("1", "2"):
        bench(items, inflight)
os.environ["PG_ITEMS"] = "1"
start_camera_server()                                     # back to the demo setting

In [ ]:
# M-2. Print the same measurement for the LAPTOP (the real internet path). Run the printed lines in a laptop
# terminal from pharmguard-camera/server (its .venv has test_client's dependencies). Do not share the key.
print(f"cd pharmguard-camera\\server")
for inflight in ("1", "2"):
    print(f".\\.venv\\Scripts\\python test_client.py --url {TUNNEL_URL.replace('https://', 'wss://')} "
          f"--key {PG_ACCESS_KEY} --frames 100 --in-flight {inflight} --quiet")
print("\nThen stream from the iPhone for 30 s and re-run O2-11: send its fps and ms/frame line too.")

---

## 10. Troubleshooting & Common Questions

### 1. GPU Unavailable (`CUDA Available: False`)
- **Cause**: Google Colab started in CPU runtime mode.
- **Solution**: From the Colab top menu, select **Runtime > Change runtime type > Hardware accelerator > T4 GPU** and click **Save**. Re-run from Section 2.

### 2. Gradio JSON-Schema Warning or Error
- **Cause**: Mismatched `pydantic` or `jsonschema` library versions in Colab.
- **Solution**: Run `!pip install -q --upgrade "pydantic>=2.5.0" "jsonschema>=4.19.0"` and restart the session (**Runtime > Restart session**).

### 3. Webcam Permission Denied in Browser
- **Cause**: Browser security policy or conflicting application.
- **Solution**: Modern browsers strictly require **HTTPS** for camera access. Always open the public `https://xxxx.gradio.live` link (never an unencrypted `http://` link). Ensure no other application (Zoom, Teams, Skype) is currently using your webcam.

### 4. Public Gradio Live Tunnel Disconnected
- **Cause**: Network timeout or Colab cellular/Wi-Fi interruption.
- **Solution**: Stop Section 8 by clicking the Stop button (or **Runtime > Interrupt execution**), wait 5 seconds, and re-run Section 8 to generate a fresh link.

### 5. Authentication Failure
- **Cause**: Incorrect username or password entered.
- **Solution**: Check the console output of Section 8 for the generated one-time credentials, or configure `PHARMAGUARD_AUTH_USER` and `PHARMAGUARD_AUTH_PASS` in Section 5.

### 6. Model Weight Download Error
- **Cause**: Intermittent network issue downloading `yolov8n.pt` from GitHub.
- **Solution**: Re-run Section 6 to re-trigger download.


---

## 11. Shutdown and Cleanup

When your surveillance demonstration is finished:

1. **Stop the Application**: Click the **Interrupt** (Stop) button on the code cell running Section 8, or press `Ctrl + M I` (or `Cmd + M I` on macOS).
2. **Release Colab GPU Resources**: To conserve your Google Colab compute units, disconnect the runtime from the top menu: **Runtime > Disconnect and delete runtime**.
